## Model 0: baseline dummy classifier

First, let us download the data set from Kaggle (if we haven't already done so):

In [1]:
import os
import shutil
import sys

# Uncomment this if on Palmetto - needed for Geopy and Kagglehub
# Note you will also need to run `module load anaconda3`
#sys.path.append("/home/nwest3/.local/lib/python3.11/site-packages")

import kagglehub
import numpy as np
import pandas as pd

if not os.path.isdir("nlp-getting-started"):
    path = kagglehub.competition_download("nlp-getting-started")
    shutil.move(path, "nlp-getting-started")

df_train = pd.read_csv("nlp-getting-started/train.csv")
df_test = pd.read_csv("nlp-getting-started/test.csv")

X_train = df_train.drop(columns=["target"])
y_train = df_train["target"]
X_test = df_test # No corresponding target column

Now we can see the data:

In [2]:
df_train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [3]:
df_test

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...


Now that that's done, let's create a dummy classifier with the default strategy for this data:

In [4]:
from sklearn.dummy import DummyClassifier

pipe = DummyClassifier()
pipe.fit(X_train, y_train)
y_test = pipe.predict(df_test)

df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
df_result.set_index("id", inplace=True)
df_result.to_csv("0_dummy.csv")

Uploading this to Kaggle gives us a score of 57.033% - approximately a coin-flip. This would indicate that the test data is fairly balanced. And if we analyze the training data, we find that it is as well.

In [5]:
y_train.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

With that in mind, there will be no need to use F1 scores or other complicated metrics when performing cross validation or grid searches later on.

## Model 1: Simple bag-of-words with Random Forest

Now, let's train an actual model on this data. First, let's create a bag-of-words vectorizer for the tweets. By default, the Scikit-Learn tokenizer for `CountVectorizer` is rather simplistic, and performs no stemming nor is it aware of things like URLs. To that end, we'll create a custom one based on the NLTK `SnowballStemmer` and `TweetTokenizer`:

In [6]:
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("stopwords") # ICYNI

STEMMER = SnowballStemmer(language = "english")
TOKENIZER = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
URL_RE = re.compile(r"^(https?://|www\.)")
HAS_AZ_RE = re.compile(r"[a-z]")

def stem_tokenize(text):
    tokens = TOKENIZER.tokenize(text)
    stems = []
    
    for tok in tokens:
        #remove urls
        if URL_RE.match(tok):
            continue

        # This removes the #, I am not sure if we want to remove so the words in # get tokenized like normal,
        # or if we want to some way signify that there is a # in there
        tok = tok.lstrip("#")

        # must have char a-z
        if not HAS_AZ_RE.search(tok):
            continue

        stems.append(STEMMER.stem(tok))

    return stems

base_stop_words = set(stopwords.words("english"))

keep_words = {
    'not', 'no', 'nor',
    'up', 'down', 'out', 'off',
    'after', 'before', 'during',
    'again', 'more', 'all',
    'over', 'only'
}
base_stop_words -= keep_words

STOP_WORDS = sorted({stop_w for words in base_stop_words for stop_w in stem_tokenize(words)})

bow_vectorizer = CountVectorizer(
    tokenizer=stem_tokenize,
    token_pattern=None,      
    lowercase=False,    
    stop_words=STOP_WORDS
)

[nltk_data] Downloading package stopwords to /home/nwest3/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Now that we have our vectorizer, let's train and cross-validate a simple Random Forest model with the default hyperparameters on it. For now, we'll ignore the keyword and location columns.

In [7]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

#ct = make_column_transformer(
#    (bow_vectorizer, "text"),
#    ("drop", ["id", "keyword", "location"])
#)
#pipe = make_pipeline(ct, RandomForestClassifier(random_state=123))

#pd.DataFrame(cross_validate(pipe, X_train, y_train, cv=5, return_train_score=True)).T

As we can see from the cross-validation results, we're overfitting _a lot_, (the trees in the forest are currently permitted unlimited complexity) but that's fine, right now, we only want a model better than the dummy classifier. Let's go ahead and send it to Kaggle.

In [8]:
#pipe.fit(X_train, y_train)
#y_test = pipe.predict(X_test)

#df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
#df_result.set_index("id", inplace=True)
#df_result.to_csv("1_randomforest.csv")

Uploading this to Kaggle gives us a score of 79.282% - Much better than the dummy model, but we could probably do better if we tuned the hyperparameters.

## Model 2: Simple bag-of-words with Neural Network

The classifier in use here could just as easily be swapped to any other model. Let's try an artificial neural network with the default parameters:

In [9]:
from sklearn.neural_network import MLPClassifier

#ct = make_column_transformer(
#    (bow_vectorizer, "text"),
#    ("drop", ["id", "keyword", "location"])
#)
#pipe = make_pipeline(ct, MLPClassifier(random_state=123))

#pd.DataFrame(cross_validate(pipe, X_train, y_train, cv=5, return_train_score=True)).T

Note that the default parameters here are overfitting as well, and it's also taking a _long_ time to fit. We'll need to do something to reduce the dimensionality of the vectorizer. Nonetheless, let's see what Kaggle thinks:

In [10]:
#pipe.fit(X_train, y_train)
#y_test = pipe.predict(X_test)

#df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
#df_result.set_index("id", inplace=True)
#df_result.to_csv("2_neuralnetwork.csv")

Uploading this to Kaggle gives us a score of 76.463%, marginally worse than the Random Forest attempt. Note that there are several features and hyperparameters we're not taking advantage of, which we'll get into for the next few models.

## Model 3: Improved bag-of-words with Random Forest

There are several problems with the previous models. The first one that we want to tackle is that the "keyword" and "location" columns aren't being utilized. Let's fix that.

First, the keywords: an HTTP-escaped string denoting some especially important feature of the tweet. There are two obvious ways to handle this. The first is to simply one-hot encode the column. Let's count how many unique values there are:

In [11]:
len(X_train["keyword"].value_counts())

221

Hm. That'd be more features than some of our dimensionality reductions for the tweets - not ideal. The second method is to simply append the keyword to the tweet. This would have the effect of weighing it twice as much in the BOW encoding. That sounds good, but unfortunately, Scikit-Learn doesn't have a feature that lets us do this easily. So we'll define our own transformer:

In [12]:
import builtins
import urllib

from sklearn.base import BaseEstimator, TransformerMixin

class KeywordJoiner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.unescape_cols = {"keyword"}

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        parts = []
        
        for colname, series in X.items():
            series = series.fillna("").astype(str)
            
            if colname in self.unescape_cols:
                series = series.apply(urllib.parse.unquote)
                
            parts.append(series)
            
        return pd.concat(parts, axis=1).agg(" ".join, axis=1).str.strip()

# Evil hack to allow this transformer to be used with multiprocessing GridSearchCV
KeywordJoiner.__module__ = "builtins"
builtins.KeywordJoiner = KeywordJoiner

KeywordJoiner().fit_transform(X_train[["text", "keyword"]])

0       Our Deeds are the Reason of this #earthquake M...
1                  Forest fire near La Ronge Sask. Canada
2       All residents asked to 'shelter in place' are ...
3       13,000 people receive #wildfires evacuation or...
4       Just got sent this photo from Ruby #Alaska as ...
                              ...                        
7608    Two giant cranes holding a bridge collapse int...
7609    @aria_ahrary @TheTawniest The out of control w...
7610    M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...
7611    Police investigating after an e-bike collided ...
7612    The Latest: More Homes Razed by Northern Calif...
Length: 7613, dtype: object

Now, for the location. It'd be nice if we could get rough latitude/longitude coordinates out of the city and country names given by Twitter users in their bio. Luckily, [Geopy](https://geopy.readthedocs.io/en/stable/) does just that by looking up an address/city/country via some online map backend (For this project, we'll use [Nominatim](https://nominatim.org/), an OpenStreetMap-based free online API). Let's make a transformer for this one too:

In [13]:
import gzip
import json

from geopy.geocoders import Nominatim

class GeopyLocator(BaseEstimator, TransformerMixin):
    def __init__(self, cachefile="geopy_cache.json.gz", backend=Nominatim):
        self.cachefile = cachefile
        self.backend = backend 
        
        try:
            with gzip.open(self.cachefile, "rt") as file:
                self.cache = json.load(file)
        except FileNotFoundError:
            self.cache = {}
            
        self.geolocator_ = None
    
    def fit(self, X, y=None):
        self.geolocator_ = self.backend(user_agent="ece_6420_final", timeout=10)
        return self

    def transform(self, X, y=None):
        locations = []
        dirty = False
        
        try:
            for x in X:
                if x != x: # NaN
                    locations.append([np.nan, np.nan])
                elif x in self.cache:
                    if self.cache[x] is None:
                        locations.append([np.nan, np.nan])
                    else:
                        locations.append(self.cache[x])
                else:
                    print(f"Geolocating \"{x}\"...")
                    
                    if x.strip(): # Special case: geolocator crashes on all-whitespace strings
                        location = self.geolocator_.geocode(x)
                    else:
                        location = None
                    dirty = True
                    time.sleep(1.5) # Respect Nominatim usage policy

                    if location is None:
                        self.cache[x] = None
                        locations.append([np.nan, np.nan])
                    else:
                        self.cache[x] = [location.latitude, location.longitude]
                        locations.append([location.latitude, location.longitude])
                        
            return np.array(locations)
        finally:
            if dirty:
                with gzip.open(self.cachefile, "wt") as file:
                    json.dump(self.cache, file)
                    
    def __getstate__(self):
        # Geopy locators aren't pickle-able, so discard them here
        self.geolocator_ = None
        return self.__dict__

GeopyLocator.__module__ = "builtins"
builtins.GeopyLocator = GeopyLocator

GeopyLocator().fit_transform(X_train["location"][:50])

array([[        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,

Note that we are using a free, rate-limited API here - if we didn't fetch this information ahead-of-time, it would take **hours** to preprocess the data set. Instead of doing that, I recommend you just download our completed cache.

In [14]:
import hashlib

h = hashlib.md5()
md5sum = "447978d49bbf2d5c044be5a6f795d9c4"

with open("geopy_cache.json.gz", "rb+") as file:
    while (chunk := file.read(1 << 16)):
        h.update(chunk)

    if h.hexdigest() != md5sum:
        h = hashlib.md5()
        file.seek(0)
        
        with urllib.request.urlopen("https://raw.githubusercontent.com/thehashtableslasher/ece6420_final/refs/heads/main/geopy_cache.json.gz") as file2:
            while (chunk := file2.read(1 << 16)):
                file.write(chunk)
                h.update(chunk)

        file.truncate()
        
        assert h.hexdigest() == md5sum

Now that we've handled those two columns, it's time to move on to improving our text representation. Right now, we're using `CountVectorizer` with default parameters, so our feature set is maximized and our final dimensionality is huge. We can reduce the dimensionality with the `max_features` hyperparameter, and we _will_ use that later, but be can also reduce the dimensionality by mapping our vectors to a more compact feature space. At first, we tried PCA, but Scikit-Learn's `PCA` actually throws a `TypeError` if you use it with sparse matrices. So instead we'll use `TruncatedSVD` - This implements Latent Semantic Indexing (LSI), a method related to PCA that is often recommended for sparse text embeddings.

With all that having been said, let's combine all our new steps into a `ColumnTransformer`:

In [15]:
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

ct = make_column_transformer(
    (make_pipeline(KeywordJoiner(), bow_vectorizer, TruncatedSVD(random_state=123)), ["text", "keyword"]),
    (make_pipeline(GeopyLocator(), StandardScaler(), SimpleImputer(strategy="constant", fill_value=0.0)), "location"),
    ("drop", ["id"])
)

ct

ColumnTransformer(transformers=[('pipeline-1',
                                 Pipeline(steps=[('keywordjoiner',
                                                  KeywordJoiner()),
                                                 ('countvectorizer',
                                                  CountVectorizer(lowercase=False,
                                                                  stop_words=['a',
                                                                              'about',
                                                                              'abov',
                                                                              'against',
                                                                              'ain',
                                                                              'am',
                                                                              'an',
                                                                              'and',
                                                                              'ani',
                                                                              'are',
                                                                              'aren',
                                                                              "aren't",
                                                                              'as',
                                                                              'at',
                                                                              'be',
                                                                              'becaus',
                                                                              'been',
                                                                              'below',
                                                                              'between',
                                                                              'both',
                                                                              'but',
                                                                              'by',
                                                                              'can',
                                                                              'couldn',
                                                                              "couldn't",
                                                                              'd',
                                                                              'did',
                                                                              'didn',
                                                                              "d...
                                                                              'do', ...],
                                                                  token_pattern=None,
                                                                  tokenizer=<function stem_tokenize at 0x7f59780b5b20>)),
                                                 ('truncatedsvd',
                                                  TruncatedSVD(random_state=123))]),
                                 ['text', 'keyword']),
                                ('pipeline-2',
                                 Pipeline(steps=[('geopylocator',
                                                  GeopyLocator()),
                                                 ('standardscaler',
                                                  StandardScaler()),
                                                 ('simpleimputer',
                                                  SimpleImputer(fill_value=0.0,
                                                                strategy='constant'))]),
                                 'location'),
                                ('drop', 'drop', ['id'])])

Additionally, we ought to tune the hyperparameters of both the model and the tweet vectorization. Random Forest gives us quite a few hyperparameters, so let's use `RandomizedSearchCV`:

In [16]:
from pprint import pp
from joblib import parallel_backend
from sklearn.model_selection import RandomizedSearchCV

N_ITER = 1 # May want to decrease if running locally

# Connor: Add params here as you see fit
param_grid = {
    "columntransformer__pipeline-1__countvectorizer__max_features": [1000, 10000, 100000, 1000000],
    "columntransformer__pipeline-1__truncatedsvd__n_components": [50, 100, 250, 500, 1000],
    "randomforestclassifier__n_estimators": [25, 50, 100, 250],
    "randomforestclassifier__criterion": ["gini", "entropy"],
    "randomforestclassifier__max_features": ["sqrt", "log2"],
    "randomforestclassifier__max_depth": [2, 3, 5, 8, 13, None]
}

    
# Note: you will need to change "multiprocessing" to "threading" on Windows
#with parallel_backend("multiprocessing", n_jobs=-1):
#if True:
#    cv = RandomizedSearchCV(make_pipeline(ct, RandomForestClassifier(random_state=123)), param_grid, n_iter=N_ITER, cv=5, return_train_score=True, random_state=123)
#    cv.fit(X_train, y_train)

#print("Best parameters:")
#pp(cv.best_params_)
#print(f"\nBest cross-validation score: {cv.best_score_}")

In [17]:
#pd.DataFrame({
#    "test_score": [cv.cv_results_[f"split{i}_test_score"][cv.best_index_] for i in range(5)],
#    "train_score": [cv.cv_results_[f"split{i}_train_score"][cv.best_index_] for i in range(5)],
#}).T

That's a bit better(?). Let's try it out on the test data.

In [18]:
#y_test = cv.predict(X_test)

#df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
#df_result.set_index("id", inplace=True)
#df_result.to_csv("3_randomforest_improved.csv")

TODO: Upload to Kaggle...

## Model 4: Improved bag-of-words with Neural Network

We can repeat the improvements we made to the Random Forest model with a neural network classifier as well:

In [19]:
# Connor: Add params here as you see fit
#param_grid = {
#    "columntransformer__pipeline-1__countvectorizer__max_features": [1000, 10000, 100000, 1000000],
#    "columntransformer__pipeline-1__truncatedsvd__n_components": [50, 100, 250, 500, 1000],
#    "mlpclassifier__hidden_layer_sizes": [[n] * m for n in [1000, 2500, 5000, 10000] for m in [2, 3, 5, 8, 13, 21, 34, 55, 89] if n * m < 100000],
#    "mlpclassifier__activation": ["relu", "logistic"]
#}

#with parallel_backend("multiprocessing", n_jobs=-1):
#if True:
#    cv = RandomizedSearchCV(make_pipeline(ct, MLPClassifier(random_state=123)), param_grid, n_iter=N_ITER, cv=5, return_train_score=True)
#    cv.fit(X_train, y_train

#print("Best parameters:")
#pp(cv.best_params_)
#print(f"\nBest cross-validation score: {cv.best_score_}")

In [20]:
#pd.DataFrame({
#    "test_score": [cv.cv_results_[f"split{i}_test_score"][cv.best_index_] for i in range(5)],
#    "train_score": [cv.cv_results_[f"split{i}_train_score"][cv.best_index_] for i in range(5)],
#}).T

In [21]:
#y_test = cv.predict(X_test)

#df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
#df_result.set_index("id", inplace=True)
#df_result.to_csv("4_neuralnetwork_improved.csv")

Upload to Kaggle...

## Model 5: Doc2Vec with Random Forest

So far, all of our models have used BOW to vectorize the tweets, but there are far more sophisticated document vectorization algorithms out there. For instance:

* n-gram vectorization
* TF-IDF
* Word2Vec/Doc2Vec
* Autoencoders such as BERT, GPT, etc.

About a decade ago, the industry state-of-the-art was [Doc2Vec](https://cs.stanford.edu/~quocle/paragraph_vector.pdf), an extension of the [Word2Vec](https://arxiv.org/pdf/1301.3781) algorithm used to embed words into a vector space. I won't get too into the weeds of how it works, but it tries to create a shallow (2 hidden layers) neural network that maps a word to a vector that can be combined with other word-to-vectors based on their context in the corpus. The classic example is that the vector for "king," minus the vector for "man," plus the vector for "woman," should roughly equal the vector for "king."

I say this was the state-of-the-art _a decade ago_ because this is an industry that has moved leaps and bounds since then, and I'm fairly sure that autoencoder-based language models like BERT have superseded Doc2Vec in most capacities. Nonetheless, let's see if we can make a good machine learning model using it. Doc2Vec isn't present in Scikit-Learn, but it is in [Gensim](https://radimrehurek.com/gensim/), so let's make one last custom transformer:

In [22]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

class D2VVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, epochs=10, tokenizer=stem_tokenize):
        self.vector_size = vector_size
        self.window = window
        self.epochs = epochs
        self.tokenizer = tokenizer
        
        self.model_ = None
    
    def fit(self, X, y=None):
        self.model_ = Doc2Vec(
            [TaggedDocument(self.tokenizer(x), [i]) for i, x in enumerate(X)],
            vector_size=self.vector_size,
            window=self.window,
            epochs=self.epochs
        )
        return self

    def transform(self, X, y=None):
        return np.vstack([self.model_.infer_vector(self.tokenizer(x)) for x in X])

D2VVectorizer.__module__ = "builtins"
builtins.D2VVectorizer = D2VVectorizer

Same song, second verse, the only difference is that we're replacing the BOW/LSI pipeline with the Doc2Vec. Note that we're dropping the keyword column now - unlike BOW, word ordering is meaningful with Doc2Vec. Perhaps future work could incorporate this data in a meaningful way.

In [23]:
ct = make_column_transformer(
    (D2VVectorizer(), "text"),
    (make_pipeline(GeopyLocator(), StandardScaler(), SimpleImputer(strategy="constant", fill_value=0.0)), "location"),
    ("drop", ["id", "keyword"])
)

N_ITER = 50

param_grid = {
    "columntransformer__d2vvectorizer__vector_size": [50, 100, 250, 500, 1000],
    "randomforestclassifier__n_estimators": [25, 50, 100, 250],
    "randomforestclassifier__criterion": ["gini", "entropy"],
    "randomforestclassifier__max_features": ["sqrt", "log2"],
    "randomforestclassifier__max_depth": [2, 3, 5, 8, 13, None]
}

with parallel_backend("multiprocessing", n_jobs=-1):
    cv = RandomizedSearchCV(make_pipeline(ct, RandomForestClassifier(random_state=123)), param_grid, n_iter=N_ITER, cv=5, return_train_score=True, random_state=123)
    cv.fit(X_train, y_train)

print("Best parameters:")
pp(cv.best_params_)
print(f"\nBest cross-validation score: {cv.best_score_}")

Best parameters:
{'randomforestclassifier__n_estimators': 100,
 'randomforestclassifier__max_features': 'log2',
 'randomforestclassifier__max_depth': 13,
 'randomforestclassifier__criterion': 'entropy',
 'columntransformer__d2vvectorizer__vector_size': 100}

Best cross-validation score: 0.7069513193667316


In [24]:
pd.DataFrame({
    "test_score": [cv.cv_results_[f"split{i}_test_score"][cv.best_index_] for i in range(5)],
    "train_score": [cv.cv_results_[f"split{i}_train_score"][cv.best_index_] for i in range(5)],
}).T

,0,1,2,3,4
test_score,0.704531,0.694025,0.701904,0.673456,0.760841
train_score,0.780460,0.786207,0.792775,0.786570,0.776063


That looks way better! Let's send a prediction to Kaggle.

In [27]:
#y_test = cv.predict(X_test)

#df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
#df_result.set_index("id", inplace=True)
#df_result.to_csv("5_randomforest_doc2vec.csv")

TODO: Upload to Kaggle

## Model 6: Doc2Vec with Neural Network

Last but not least, let's see how a neural network fares with Doc2Vec.